# SentinelMail — Rule-Based DLP Benchmark

Evaluates `models/rule_based/RuleBasedDetector` (Microsoft Presidio) against the
ai4privacy validation split as a zero-training baseline for the ensemble benchmark.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

REPO_ROOT = Path("..").resolve()
sys.path.insert(0, str(REPO_ROOT))

DATA_RAW = REPO_ROOT / "data" / "ai4privacy" / "raw"
RESULTS_DIR = REPO_ROOT / "evaluation" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

FINPII_TYPES = {
    "ACCOUNTNUMBER", "CREDITCARDNUMBER", "CREDITCARDCVV", "CREDITCARDISSUER",
    "IBAN", "SWIFT", "CURRENCY", "CURRENCYNAME", "CURRENCYCODE", "CURRENCYSYMBOL",
    "AMOUNT", "TAXNUMBER", "SOCIALSECURITYNUMBER", "SORTCODE", "ETHEREUMADDRESS",
    "BITCOINADDRESS", "LITECOINADDRESS",
}
LABEL_COLS = ["benign", "PII", "financial", "health", "confidential"]

## 1. Load Validation Split

In [ ]:
ds = load_dataset("ai4privacy/pii-masking-300k", cache_dir=str(DATA_RAW))
val_en = ds["validation"].filter(lambda x: x["language"] == "English")
val_df = val_en.to_pandas()
print(f"Validation rows (English): {len(val_df):,}")

## 2. Build Ground-Truth Labels

In [ ]:
def parse_mask(mask):
    if mask is None or (isinstance(mask, float)):
        return []
    if isinstance(mask, np.ndarray):
        return list(mask)
    if isinstance(mask, list):
        return mask
    if isinstance(mask, str):
        try:
            return json.loads(mask)
        except (json.JSONDecodeError, TypeError):
            return []
    return []

def make_labels(mask):
    entities = parse_mask(mask)
    has_pii = len(entities) > 0
    has_fin = any(e["label"] in FINPII_TYPES for e in entities)
    return {
        "benign": int(not has_pii),
        "PII": int(has_pii),
        "financial": int(has_fin),
        "health": 0,
        "confidential": 0,
    }

label_df = val_df["privacy_mask"].apply(make_labels).apply(pd.Series)
val_df = pd.concat([val_df, label_df], axis=1)

print("Ground-truth label distribution (validation, EN):")
print(val_df[LABEL_COLS].sum().to_string())

## 3. Run Rule-Based Detector

In [ ]:
from models.rule_based import RuleBasedDetector

detector = RuleBasedDetector(score_threshold=0.4)
print("Detector loaded. Running on validation set...")

predictions = []
for text in tqdm(val_df["source_text"], desc="Detecting"):
    predictions.append(detector.predict(text))

pred_df = pd.DataFrame(predictions)
print("Predicted label distribution:")
print(pred_df[LABEL_COLS].sum().to_string())

## 4. Per-Label Metrics

In [ ]:
# Evaluate only on labels with ground-truth coverage (skip health/confidential — all zeros)
eval_labels = [l for l in LABEL_COLS if val_df[l].sum() > 0]

metrics: dict = {}
for label in eval_labels:
    y_true = val_df[label].values
    y_pred = pred_df[label].values
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    metrics[label] = {
        "precision": round(report["1"]["precision"], 4),
        "recall":    round(report["1"]["recall"],    4),
        "f1":        round(report["1"]["f1-score"],  4),
        "support":   int(report["1"]["support"]),
    }
    print(f"\n--- {label} ---")
    print(classification_report(y_true, y_pred, zero_division=0))

## 5. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, len(eval_labels), figsize=(5 * len(eval_labels), 4))
if len(eval_labels) == 1:
    axes = [axes]

for ax, label in zip(axes, eval_labels):
    cm = confusion_matrix(val_df[label].values, pred_df[label].values)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["pred 0", "pred 1"], yticklabels=["true 0", "true 1"])
    ax.set_title(f"{label}  (F1={metrics[label]['f1']:.3f})")

plt.suptitle("Rule-Based Detector — Confusion Matrices (validation EN)", y=1.02)
plt.tight_layout()
plt.show()

## 6. Error Analysis

In [ ]:
N = 10

for label in eval_labels:
    fp_idx = val_df.index[(val_df[label] == 0) & (pred_df[label] == 1)]
    fn_idx = val_df.index[(val_df[label] == 1) & (pred_df[label] == 0)]

    print(f"\n=== {label.upper()} — False Positives (n={len(fp_idx)}) ===")
    for i in fp_idx[:N]:
        text = val_df.loc[i, "source_text"]
        hits = detector.explain(text)
        print(f"  text[:50]: {text[:50]}…")
        print(f"  detections: {[(h['entity_type'], h['text_snippet']) for h in hits]}")

    print(f"\n=== {label.upper()} — False Negatives (n={len(fn_idx)}) ===")
    for i in fn_idx[:N]:
        text = val_df.loc[i, "source_text"]
        entities = [(e["label"], str(e["value"])[:20]) for e in parse_mask(val_df.loc[i, "privacy_mask"])]
        print(f"  text[:50]: {text[:50]}…")
        print(f"  ground-truth entities: {entities}")

## 7. Save Results

In [ ]:
output = {
    "model": "rule_based_presidio",
    "dataset": "ai4privacy/pii-masking-300k",
    "split": "validation",
    "language_filter": "English",
    "score_threshold": detector.score_threshold,
    "n_samples": len(val_df),
    "per_label": metrics,
}

out_path = RESULTS_DIR / "rule_based_metrics.json"
with open(out_path, "w") as f:
    json.dump(output, f, indent=2)

print(f"Saved to {out_path}")
print(json.dumps(output, indent=2))